#### 1-Librerias

## 1.1-Instalacion librerias externas

In [26]:
#!pip install /kaggle/input/noisereduce/noisereduce-3.0.3-py3-none-any.whl

## 1.2-Importar librerias

In [27]:
#import numpy as np # linear algebra
import pandas as pd # data processing, CSV file 
import os # creating directories
import shutil # copiar archivos
import random

# cleaning audio
#import noisereduce as nr
from scipy.signal import butter, filtfilt, sosfiltfilt
import librosa

import os, glob, gc, numpy as np
from pathlib import Path
from tqdm import tqdm

import torch.nn as nn
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import models
from collections import defaultdict
from torchvision import models as tv

from sklearn.metrics import roc_auc_score

import timm

#import torch.serialization
#torch.serialization.add_safe_globals([np.core.multiarray.scalar])

import torch.serialization
torch.serialization.add_safe_globals([np.core.multiarray.scalar])

/tmp/ipykernel_58/2260480713.py:32: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  torch.serialization.add_safe_globals([np.core.multiarray.scalar])


## 1.3-Rutas y directorios

In [28]:
def create_folder(path, carpeta):
    folder = os.path.join(path, carpeta)
    os.makedirs(folder, exist_ok=True)
    print(f'[+] Folder "{carpeta}" created.')
    return folder

def get_path(dir, filename):
    return os.path.join(dir, filename)

def clean_folder(path, delete_folder=False):
    if delete_folder:
        shutil.rmtree(path)
        print(f"[+] Carpeta eliminada: {path}")
    else:
        for filename in os.listdir(path):
            file_path = os.path.join(path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.remove(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"Error eliminando {file_path}: {e}")
        print(f"[+] Contenido eliminado en: {path}")

In [29]:
#  birdclef en /kaggle/input/birdclef-2025
path_birdclef = '/kaggle/input/competitions/birdclef-2025'
path_home = '/kaggle/working'


path_models = create_folder(path_home, "models")

path_train_audio = get_path(path_birdclef, "train_audio")

path_train_soundscapes = get_path(path_birdclef, "train_soundscapes")

path_index_test = "/kaggle/input/datasets/dprieeto/model-epochs/index_test.csv"

# leer csv test
index_test = pd.read_csv(path_index_test)
index_test["filename"] = index_test["filename"].astype(str)
index_test["primary_label"] = index_test["primary_label"].astype(str)

local_labels = index_test["primary_label"].tolist()

print(path_index_test)
print(path_train_soundscapes)

path_taxonomy_csv = get_path(path_birdclef, "taxonomy.csv")

path_sample_submission = get_path(path_birdclef, "sample_submission.csv")

path_data = create_folder(path_home, "data")

[+] Folder "models" created.
/kaggle/input/datasets/dprieeto/model-epochs/index_test.csv
/kaggle/input/competitions/birdclef-2025/train_soundscapes
[+] Folder "data" created.


# 2-Preprocesamiento

## 2.1- Parametros

In [30]:
### Parametros modificables ###
# Audio / STFT Config
SAMPLE_RATE = 32000 # Tasa de muestreo, formato del dataset
N_FFT = 1024 # Transformada de Fourier
HOP_LENGTH = 320 # nº muestras entre el inicio de ventanas, ver stft
F_MIN = 20 # Frecuencia minima Hz
F_MAX = 16000 # Frecuencia maxima Hz
MIN_SEG_DURATION_SEC = 0.5 # Duracion minima para intervalos

# Mels
N_MELS = 128 # Nº de bandas del espectograma de mel, reducido 128->64 problemas de espacio
MEL_DURATION = 5.0  # Duracion en segundos por segmento

# VAD Recortes
TOP_DB = None #30# Umbral para detectar actividad, <30 = silencio

# Filtros de banda
APPLY_BANDPASS = True #False
LOWCUT = 150 #255 #300  # Frecuencia de corte inferior
HIGHCUT = 15550 #9000 #15950  # Frecuencia de corte superior

# Pipelina Preprocesado
FLAG_PROCESS= False # True si se quiere preprocesar los datos
FLAG_MAP = False # True si se quiere mapear el train csv
FLAG_LOAD_CSV = False# True si se quiere cargar desde un dataset externo index train, val trina, index audio
RNG_SEED = 42 # semilla para muestreo

## 2.2-Fuciones limpieza y preprocesado

In [31]:
# Funciones

def load_audio(filepath, sr=SAMPLE_RATE):
    # Cargar el archivo de audio
    y, sr = librosa.load(filepath, sr=sr, mono=True, dtype=np.float32, res_type='kaiser_fast')
    return y, sr


def reduce_noise(y, sr):
    # Reducir el ruido
    y_denoised = nr.reduce_noise(y=y, sr=sr)
    return y_denoised


def bandpass_filter(y, sr, lowcut=LOWCUT, highcut=HIGHCUT, order=5):
    if not APPLY_BANDPASS:
        return y
    # Filtrado para eliminar frecuencias irrelevantes (Filtro paso bajo y filtro paso alto)
    nyquist = 0.5 * sr
    low = max(1.0, float(lowcut))
    high = min(float(highcut), nyquist * 0.98)
    if low >= high: # datos invalidos
        return y
    low = lowcut / nyquist
    high = highcut / nyquist
    # Butterworth formato sos y filtrado cero-fase
    sos= butter(order, [low, high], btype='band', output='sos')
    y_filtered = sosfiltfilt(sos, y) # evitar desfase
    return y_filtered


def normalize_audio(y):
    # Normalizacion la señal del audio, par aque todos los audios tengan un volumen similar
    # (ya sea entre -1 y 1)
    y_normalized = librosa.util.normalize(y)
    return y_normalized


def detect_active_segments(y, sr, top_db=TOP_DB):
    if top_db is None:
        return np.array([[0, len(y)]], dtype=int)
    # devuelve array de pares [start, end]
    return librosa.effects.split(y, top_db=top_db)

def preprocess_wave(y, sr, use_bp=False, do_norm=True, denoise_fn=None):
    if denoise_fn is not None:
        y = denoise_fn(y=y, sr=sr)
    if use_bp:
        y = bandpass_filter(y, sr)
    if do_norm:
        y = normalize_audio(y)
    return y

In [32]:
def extract_mel(y_segment, sr,
                n_mels=N_MELS,
                hop_length=HOP_LENGTH,
                duration=MEL_DURATION,
                n_fft=N_FFT,
                fmin=F_MIN,
                fmax=F_MAX):
    """
    - Ajusta el segmento a 'duration' segundos (pad/cut).
    - Calcula Mel con tus hiperparámetros globales.
    - Usa log1p(mel) para estabilidad numérica y mejor comportamiento en training.
    - Devuelve [n_mels, n_frames] en float32.
    """
    # Ajuste exacto de longitud
    samples = int(duration * sr)
    y_fixed = librosa.util.fix_length(y_segment, size=samples)

    # Mel spectrogram (power=2.0 → espectro de potencia)
    mel = librosa.feature.melspectrogram(
        y=y_fixed,
        sr=sr,
        n_mels=n_mels,
        hop_length=hop_length,
        n_fft=n_fft,
        fmin=fmin,
        fmax=fmax,
        power=2.0
    )

    # Log-mel estable
    mel = np.log1p(mel).astype(np.float32)
    # z-score
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel

## 2.3-Funcion principal
Aplica todas las funciones anteriores del apartado 2.2

In [33]:
def preprocess_audio(filepath, noised=False):
    y, sr = load_audio(filepath)
    if noised:
        y = reduce_noise(y, sr)
    y = bandpass_filter(y, sr)
    y = normalize_audio(y)

    intervals = detect_active_segments(y, sr)

    mel_segments = []
    for start, end in intervals:
        segment = y[start:end]

        # si la duracion es menor a un segundo
        if (end - start) < sr*MIN_SEG_DURATION_SEC:
            continue  # descarta el segmento

        if is_likely_speech(segment, sr):
            continue

        mel = extract_mel(segment, sr, n_mels=64, hop_length=HOP_LENGTH)

        if mel is not None:
            mel_segments.append(mel)

    return mel_segments  # Lista de [128, fijo]

## 2.4-Aplicar preprocesado

In [34]:
# Cargar taxonomy 
"""
taxonomy = pd.read_csv(path_taxonomy_csv, encoding='ISO-8859-1')
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str).str.strip()
valid_labels = set(taxonomy['primary_label'].unique())
print(f"[+] Especies taxonomy:{len(valid_labels)}")
label_to_index = {label: idx for idx, label in enumerate(valid_labels)}
"""
sample = pd.read_csv(path_sample_submission)
CLASSES = list(sample.columns[1:])  # orden oficial

label_to_index = {lab: i for i, lab in enumerate(CLASSES)}
index_to_label = {i: lab for lab, i in label_to_index.items()}

# preparar carpeta donde se guarda el preprocesado
#clean_folder(path_data)
mel_dir = create_folder(path_data, "mels")
print(mel_dir)

def get_local_test_files(index_test, audio_root):
    filenames = index_test["filename"].astype(str).tolist()
    wav_files = [Path(audio_root) / fname for fname in filenames]
    return wav_files, filenames

[+] Folder "mels" created.
/kaggle/working/data/mels


In [35]:
# Parametros
NPZ_OUT    = Path(get_path(mel_dir, "test_mels.npz"))             
SR         = 32000
SEG_SEC    = 5                                                  # duración de cada trozo
MAX_BASE_WINDOWS = 12
RNG_SEED = 42

# Buscar archivos
wav_files, local_filenames = get_local_test_files(index_test, path_train_audio)

missing_files = [str(p) for p in wav_files if not Path(p).exists()]
if len(missing_files) > 0:
    print(f"[WARN] No se encontraron {len(missing_files)} archivos.")
    print(missing_files[:5])
else:
    print(f"[+] Todos los archivos del test local existen: {len(wav_files)}")

                                          # usa el mismo que en entrenamiento

# ----------------- 2. helpers (sin etiquetas) -------------------------------
def slice_audio_augmented(y, sr, seg=SEG_SEC, stride_sec=5, max_windows=MAX_BASE_WINDOWS, seed=RNG_SEED):
    """
    Genera ventanas de 5 segundos sobre la duración real del audio.
    Para cada ventana base genera 4 variantes:
    - 0-5 completa
    - 0-3
    - 1-4
    - 2-5

    Si la grabación tiene más de max_windows ventanas base,
    selecciona max_windows aleatorias de forma reproducible.
    """
    windows = []
    win_samples = int(seg * sr)
    stride_samples = int(stride_sec * sr)

    if len(y) < win_samples:
        y = librosa.util.fix_length(y, size=win_samples)

    starts = list(range(0, len(y), stride_samples))

    if len(starts) > max_windows:
        rng = np.random.default_rng(seed)
        starts = sorted(rng.choice(starts, size=max_windows, replace=False).tolist())

    offsets = [
        (0, 5),
        (0, 3),
        (1, 4),
        (2, 5),
    ]

    for start_sample in starts:
        base_seg = y[start_sample:start_sample + win_samples]
        base_seg = librosa.util.fix_length(base_seg, size=win_samples)

        for start_s, end_s in offsets:
            s = int(start_s * sr)
            e = int(end_s * sr)

            y_seg = base_seg[s:e]
            y_seg = librosa.util.fix_length(y_seg, size=win_samples)

            windows.append(y_seg)

    return windows

def preprocess_segment_to_mel(y_seg, sr, 
                              apply_bandpass=APPLY_BANDPASS):

    if apply_bandpass:
        y_seg = bandpass_filter(y_seg, sr)
    y_seg = normalize_audio(y_seg)

    mel = extract_mel(
        y_seg, sr, n_mels=N_MELS, 
        hop_length=HOP_LENGTH)
    return mel                                                     # shape [64, T] o None

[+] Todos los archivos del test local existen: 713


In [36]:
# Bucle principal test local
mel_segments_all  = []
segment_names_all = []

print("[*] Procesando archivos del test local")

for i, wav_path in enumerate(tqdm(wav_files, total=len(wav_files), desc='procesando test local')):
    file_id = local_filenames[i]

    # Cargar audio completo
    y, sr = load_audio(wav_path, sr=SR)

    # Generar ventanas aumentadas sobre la duración real
    for seg in slice_audio_augmented(y, sr, seg=SEG_SEC, stride_sec=5):
        mel = preprocess_segment_to_mel(seg, sr)
        mel_segments_all.append(mel.astype(np.float16))

        # Todas las ventanas del mismo audio comparten ID
        segment_names_all.append(file_id)


# ----------------- 4. guardado único ----------------------------------------
np.savez_compressed(
    NPZ_OUT,
    mels=np.stack(mel_segments_all),
    ids=np.array(segment_names_all)
)

print(f"[+] Listo: {len(segment_names_all)} segmentos guardados en {NPZ_OUT}")

[*] Procesando archivos del test local


procesando test local: 100%|██████████| 713/713 [18:54<00:00,  1.59s/it]


[+] Listo: 16396 segmentos guardados en /kaggle/working/data/mels/test_mels.npz


# 3-Cargar modelo

## 3.1-Configuracion modelo

In [45]:
# Configuración de modelos
NUM_CLASSES = 206
DROPOUT = 0.4
PRETRAINED = False  # En inferencia no descarga pesos; se cargan desde checkpoint
ENSEMBLE = False # True si se quiere hacer ensemble
ALPHA_ENSEMBLE = 0.35  # peso del primer modelo en ensemble
MODEL_NAME = "resnet" # prefijo del modelo cuando no aplica ensemble

if ENSEMBLE:
    MODEL_CONFIGS = [
        {
            "prefix": "effnet",
            "checkpoint": get_path("/kaggle/input/model-epochs/", "effnet_v3.0.pth"),
            "weight": ALPHA_ENSEMBLE,
        },
        {
            "prefix": "resnet",
            "checkpoint": get_path("/kaggle/input/model-epochs/", "resnet_v3.2.pth"),
            "weight": 1.0 - ALPHA_ENSEMBLE,
        },
    ]
else:
    MODEL_CONFIGS = [
        {
            "prefix": MODEL_NAME,
            "checkpoint": get_path("/kaggle/input/datasets/dprieeto/model-epochs/", "resnet_ep012.pth"),
            "weight": 1.0,
        }
    ]

# configuracion predicciones
POOL = "max"   # "mean"|"max"|"mix"|"topk"
ALPHA = 0.3 # solo afecta a 'mix'
TOPK  = 3

# Verbose resumen
MODELO = "Resnet-18"
EPOCA = 12

# configuracion inferencia
BATCH_SIZE = 8
device = "cuda"

## 3.2-Redes neuronales

### 3.2.1-BirdCNN

In [46]:
class BirdCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.3, in_channels=1):  
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.MaxPool2d((3, 2)),                                  
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),                                 
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                           
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:                      
            x = x.unsqueeze(1) # añade canal              
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

### 3.2.2-EfficientNet

In [47]:
class BirdEfficientNet(nn.Module):
    def __init__(self, num_classes, in_chans=1, pretrained=False, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            in_chans=in_chans,
            num_classes=0  # quitamos la head para añadir la nuestra con dropout
        )

        feat_dim = self.backbone.num_features
        
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        feats = self.backbone(x) # usa pesos pretrained
        logits = self.head(feats) # head aprende
        return logits

### 3.2.3-ResNet

In [48]:
class BirdResNet(nn.Module):
    def __init__(self, num_classes, in_chans=1, pretrained=False, dropout=0.3):
        super().__init__()
        if pretrained:
            try:
                base = tv.resnet18(weights=tv.ResNet18_Weights.IMAGENET1K_V1)
            except Exception:
                base = tv.resnet18(pretrained=True)
        else:
            try:
                base = tv.resnet18(weights=None)
            except Exception:
                base = tv.resnet18(pretrained=False)

        # conv1 adapt
        orig = base.conv1
        if in_chans != orig.in_channels:
            new_conv = nn.Conv2d(in_chans, orig.out_channels,
                                 kernel_size=orig.kernel_size, stride=orig.stride,
                                 padding=orig.padding, bias=False)
            if pretrained:
                with torch.no_grad():
                    w = orig.weight.data
                    if in_chans == 1:
                        new_conv.weight.copy_(w.mean(1, keepdim=True))
                    else:
                        new_conv.weight.copy_(w.mean(1, keepdim=True).repeat(1, in_chans, 1, 1))
            base.conv1 = new_conv

        # backbone = todo menos la fc
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base

        # head
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        feats = self.backbone(x)      
        logits = self.head(feats)
        return logits

## 3.3-Cargar modelo

In [49]:
def build_model(model_prefix):
    if model_prefix == "cnn":
        model = BirdCNN(NUM_CLASSES, dropout=DROPOUT)
    elif model_prefix == "effnet":
        model = BirdEfficientNet(NUM_CLASSES, dropout=DROPOUT, pretrained=PRETRAINED)
    elif model_prefix == "resnet":
        model = BirdResNet(NUM_CLASSES, dropout=DROPOUT, pretrained=PRETRAINED)
    else:
        raise ValueError(f"Modelo no reconocido: {model_prefix}")
    
    return model

def load_model_from_config(cfg):
    model_prefix = cfg["prefix"]
    path_model = cfg["checkpoint"]

    model = build_model(model_prefix)

    ckpt = torch.load(path_model, map_location="cpu", weights_only=False)

    # El script de training guardo los pesos en la clave "model_state"
    state = ckpt["model_state"]

    model.load_state_dict(state)
    model = model.to(device)
    model.eval()

    print(f"[+] Modelo cargado correctamente: {model_prefix} desde {path_model}")

    return model

In [50]:
for cfg in MODEL_CONFIGS:
    model = load_model_from_config(cfg)

[+] Modelo cargado correctamente: resnet desde /kaggle/input/datasets/dprieeto/model-epochs/resnet_ep012.pth


# 4-Inferencia

## 4.0 funcion pooling

In [51]:
def pool_preds(preds, mode="mix", alpha=0.5, topk=3):
    if preds.shape[0] == 1:
        return preds[0]
    if mode == "mean":
        return preds.mean(axis=0)
    if mode == "max":
        return preds.max(axis=0)
    if mode == "mix":
        return (1-alpha)*preds.mean(axis=0) + alpha*preds.max(axis=0)
    if mode == "topk":
        k = min(topk, preds.shape[0])
        topk_mean = np.sort(preds, axis=0)[-k:, :].mean(axis=0)
        return 0.5*preds.mean(axis=0) + 0.5*topk_mean
    raise ValueError(mode)

## 4.1-Cargar espectros

In [52]:
# Cargar test npz
data = np.load(NPZ_OUT)

# 1) Extraer arrays: 'mels' tiene forma [N, N_MELS, T], y 'ids' son los row_id
mels_np = data["mels"]    # numpy array de shape (N, N_MELS, T)
row_ids = data["ids"]     # lista de strings (tamaño N)

N, n_mels, T = mels_np.shape
print(f"[+] {N} segmentos cargados (cada uno con {n_mels} bandas Mel y {T} frames)")

# Columnas de especies
df_sample = pd.read_csv(path_sample_submission)
species_cols = df_sample.columns[1:].tolist()
NUM_CLASSES = len(species_cols)

[+] 16396 segmentos cargados (cada uno con 128 bandas Mel y 501 frames)


### 4.1.1- funcion auxiliar

In [53]:
def predict_rows_single_model(model, mels_np, batch_size=BATCH_SIZE):
    """
    Devuelve probabilidades por fila/segmento del npz.
    Shape de salida: [N, NUM_CLASSES]
    """
    N = len(mels_np)
    probs_rows = np.zeros((N, NUM_CLASSES), dtype=np.float32)

    for start in tqdm(range(0, N, batch_size), desc="Predicciones desde .npz"):
        end = min(start + batch_size, N)

        batch_mels = mels_np[start:end]  # [B, MELS, T]
        batch = (
            torch.tensor(batch_mels, dtype=torch.float32)
            .unsqueeze(1)
            .to(device)
        )  # [B, 1, MELS, T]

        with torch.no_grad():
            logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()

        probs_rows[start:end] = probs

    return probs_rows

## 4.2-Generar predicciones

In [54]:
ensemble_probs_rows = np.zeros((N, NUM_CLASSES), dtype=np.float32)
total_weight = 0.0

print("[*] Iniciando predicciones...")

for cfg in MODEL_CONFIGS:
    print(f"[*] Prediciendo con modelo: {cfg['prefix']} | peso={cfg['weight']}")

    model = load_model_from_config(cfg)

    probs_rows_model = predict_rows_single_model(
        model=model,
        mels_np=mels_np,
        batch_size=BATCH_SIZE
    )

    ensemble_probs_rows += cfg["weight"] * probs_rows_model
    total_weight += cfg["weight"]

    del model
    gc.collect()

ensemble_probs_rows /= total_weight

[*] Iniciando predicciones...
[*] Prediciendo con modelo: resnet | peso=1.0
[+] Modelo cargado correctamente: resnet desde /kaggle/input/datasets/dprieeto/model-epochs/resnet_ep012.pth


Predicciones desde .npz: 100%|██████████| 2050/2050 [00:21<00:00, 93.73it/s]


In [55]:
# Pooling por row_id (solo si hay repetidos)


s = pd.Series(row_ids)

if len(s) == s.nunique():
    # no hay repetidos -> ya es final
    probs_all = ensemble_probs_rows
    row_ids_out = row_ids
else:
    agg = defaultdict(list)

    for rid, p in zip(row_ids, ensemble_probs_rows):
        agg[rid].append(p)

    row_ids_out = list(agg.keys())
    probs_all = np.zeros((len(row_ids_out), NUM_CLASSES), dtype=np.float32)

    for i, rid in enumerate(row_ids_out):
        preds = np.stack(agg[rid], axis=0)  # [n_windows, 206]
        probs_all[i] = pool_preds(preds, mode=POOL, alpha=ALPHA, topk=TOPK)

print("[INFO] probs_all shape:", probs_all.shape)

[INFO] probs_all shape: (713, 206)


## 4.3-Guardar prediciones locales

In [56]:
assert probs_all.shape[0] == len(row_ids_out), "El número de filas no coincide"
assert probs_all.shape[1] == len(species_cols), "El número de clases no coincide"

# Construir y_true e y_pred para test local
label_map = dict( # Mapa filename -> primary_label
    zip(
        index_test["filename"].astype(str),
        index_test["primary_label"].astype(str)
    )
)

# y_pred: probabilidades agregadas por audio
y_pred = probs_all.copy()

# y_true: matriz [n_audios, 206]
y_true = np.zeros((len(row_ids_out), len(species_cols)), dtype=np.int8)

for i, file_id in enumerate(row_ids_out):
    true_label = label_map[file_id]

    if true_label in label_to_index:
        y_true[i, label_to_index[true_label]] = 1
    else:
        print(f"[WARN] Etiqueta no encontrada en species_cols: {true_label}")


# Guardar fichero completo y_true + y_pred
df_meta = pd.DataFrame({
    "filename": row_ids_out,
    "primary_label": [label_map[file_id] for file_id in row_ids_out]
})

df_true = pd.DataFrame(y_true, columns=[f"true_{c}" for c in species_cols])
df_pred = pd.DataFrame(y_pred, columns=[f"pred_{c}" for c in species_cols])

df_local = pd.concat([df_meta, df_true, df_pred], axis=1)

df_local.to_csv("predicciones_test_local.csv", index=False)
print("[+] Guardado predicciones_test_local.csv")

[+] Guardado predicciones_test_local.csv


# 5.-AUC-ROC
AUC-ROC globla, top mejores clase que generaliza y top peores

## 5.1-Obtener valor AUC-ROC global

In [58]:
# AUC-ROC macro local, estilo Kaggle
auc_por_clase = []

for j, species in enumerate(species_cols):
    y_true_j = y_true[:, j]
    y_pred_j = y_pred[:, j]

    # AUC solo se puede calcular si hay positivos y negativos
    if y_true_j.sum() > 0 and y_true_j.sum() < len(y_true_j):
        auc_j = roc_auc_score(y_true_j, y_pred_j)
        auc_por_clase.append({
            "species": species,
            "n_pos": int(y_true_j.sum()),
            "auc": auc_j
        })

df_auc = pd.DataFrame(auc_por_clase)
auc_macro_local = df_auc["auc"].mean()

print(f"[RESULT] AUC-ROC macro local: {auc_macro_local:.4f}")
print(f"[INFO] Especies evaluadas: {len(df_auc)} / {len(species_cols)}")

df_auc = df_auc.sort_values("auc", ascending=False)
df_auc.to_csv("auc_por_clase_test_local.csv", index=False)

print("[+] Guardado auc_por_clase_test_local.csv")

df_resumen_local = pd.DataFrame([{
    "modelo": MODELO,
    "epoca": EPOCA,
    "agregacion": POOL,
    "auc_macro_local": auc_macro_local,
    "especies_evaluadas": len(df_auc),
    "especies_totales": len(species_cols),
    "n_audios_test_local": len(row_ids_out)
}])

df_resumen_local.to_csv("resumen_test_local.csv", index=False)
print("[+] Guardado resumen_test_local.csv")

[RESULT] AUC-ROC macro local: 0.9545
[INFO] Especies evaluadas: 153 / 206
[+] Guardado auc_por_clase_test_local.csv
[+] Guardado resumen_test_local.csv


## 5.2.-top mejores y peores clases

In [59]:
top10_mejores = df_auc.head(10)
top10_peores = df_auc.tail(10).sort_values("auc", ascending=True)

top10_mejores.to_csv("top10_mejores_test_local.csv", index=False)
top10_peores.to_csv("top10_peores_test_local.csv", index=False)

display(top10_mejores)
display(top10_peores)

,species,n_pos,auc
1,21211,2,1.0
6,41970,1,1.0
2,22333,1,1.0
3,22973,2,1.0
4,22976,1,1.0
13,566513,1,1.0
15,65448,2,1.0
12,555086,1,1.0
10,517119,1,1.0
16,65962,1,1.0


,species,n_pos,auc
108,savhaw1,1,0.561798
61,eardov1,2,0.567511
7,476538,1,0.591292
21,anhing,1,0.603933
84,palhor2,1,0.606742
9,50186,1,0.625000
124,strowl1,4,0.667489
39,brtpar1,2,0.787623
94,recwoo1,2,0.797468
104,ruther1,2,0.808017
